# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mishellscripts/flyrank/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
url = "hf://datasets/FlyRank/internship-warehouse"
sample_rel = f"{url}/fact_content_daily_performance_sample.parquet"
rel = f"{url}/fact_content_daily_performance/**/*.parquet"

con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The method I choose to use is the "yes/no" with observed label using logistic regression. This label attempts to a answer a question of: Did a declining page recover by the next time period? This fits the refresh/content opportunity scoring lane because it studies the refresh potential of a declining page. To start, logistic regression and random forest will be used. Features will be selected according to the data contract and multicollinearity will be researched. The output model's coefficients will provide explanations to important features. Boost methods will be considered to further enhance precision.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Training can only occur on client data with enough history. A cutoff date T will be selected and tuned to train the model using future (after T) recovery data and past (before T) decline data. Clients are to be split into train/test using 10-fold cross validation.

### Picking date split

To select a cutoff date T, we need to find eligible clients (clients with at least 2 months of data before and after) for each date option. The date of interest is the date with the highest amount of eligible clients.

In [36]:
import pandas as pd

date_range = con.sql(f"""
    SELECT MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM read_parquet('{rel}')
""").df()
earliest, latest = date_range["earliest"][0], date_range["latest"][0]

candidate_Ts = pd.date_range(earliest, latest, freq="MS")

scan_results = []
for T_candidate in candidate_Ts:
    T_str = T_candidate.strftime("%Y-%m-%d")
    n = con.sql(f"""
        WITH client_ranges AS (
            SELECT client_hash_id, MIN(report_date) AS min_date, MAX(report_date) AS max_date
            FROM read_parquet('{rel}')
            GROUP BY client_hash_id
        )
        SELECT COUNT(DISTINCT CASE WHEN min_date <= (DATE '{T_str}' - INTERVAL 59 DAY)
                                    AND max_date >= (DATE '{T_str}' + INTERVAL 61 DAY)
                               THEN client_hash_id END) AS n
        FROM client_ranges
    """).df()["n"][0]
    scan_results.append({"T": T_str, "eligible_clients": n})

scan_df = pd.DataFrame(scan_results)
print(scan_df)

T = scan_df.loc[scan_df['eligible_clients'].idxmax(), 'T']
print(f"\nBest T: {T} "
      f"with {scan_df['eligible_clients'].max()} eligible clients")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

             T  eligible_clients
0   2025-02-01                 0
1   2025-03-01                 0
2   2025-04-01                 2
3   2025-05-01                 3
4   2025-06-01                 4
5   2025-07-01                 4
6   2025-08-01                 4
7   2025-09-01                10
8   2025-10-01                16
9   2025-11-01                16
10  2025-12-01                24
11  2026-01-01                32
12  2026-02-01                39
13  2026-03-01                39
14  2026-04-01                38
15  2026-05-01                 0
16  2026-06-01                 0

Best T: 2026-02-01 with 39 eligible clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [99]:
WINDOW_DAYS = 30       # comparison window size (last30/prev30 convention)
LABEL_GAP_DAYS = 1     # days between T and the start of the recovery-check window
LABEL_WINDOW_DAYS = 30 # size of the recovery-check window

# --- feature set, as of T only, no label-derived fields ---
content_type_query = f"""
    SELECT content_hash_id, keyword_char_count, url_char_count, content_type,
        search_volume, competition, main_intent, category_count, model_used, char_count,
        DATEDIFF('day', content_updated_date, DATE '{T}') AS days_since_last_update,
        DATEDIFF('day', content_created_date, DATE '{T}') AS days_since_created
    FROM read_parquet('{url}/dim_content.parquet')
"""
content_dim = con.sql(content_type_query).df()

feature_query = f"""
    SELECT client_hash_id, content_hash_id,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks
    FROM read_parquet('{rel}')
    WHERE report_date <= DATE '{T}'
    GROUP BY client_hash_id, content_hash_id
"""
X_raw = con.sql(feature_query).df().merge(content_dim, on="content_hash_id", how="left")

# --- gate: is_declining at T -- window boundaries derived entirely from T + WINDOW_DAYS ---
gate_query = f"""
    WITH windowed AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {WINDOW_DAYS - 1} DAY) AND DATE '{T}'
                     THEN gsc_impressions ELSE 0 END) AS impr_last30,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {2*WINDOW_DAYS - 1} DAY)
                                           AND (DATE '{T}' - INTERVAL {WINDOW_DAYS} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_prev30
        FROM read_parquet('{rel}')
        WHERE report_date <= DATE '{T}'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
        CASE WHEN (impr_last30 - impr_prev30) / NULLIF(impr_prev30, 0) * 100 < -20
             THEN 1 ELSE 0 END AS is_declining_at_T
    FROM windowed
"""
gate_df = con.sql(gate_query).df()

# --- recovery label: strictly after T -- also fully derived ---
future_query = f"""
    WITH future AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + LABEL_WINDOW_DAYS} DAY)
                                           AND (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + 2*LABEL_WINDOW_DAYS - 1} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_T1,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS} DAY)
                                           AND (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + LABEL_WINDOW_DAYS - 1} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_baseline
        FROM read_parquet('{rel}')
        WHERE report_date > DATE '{T}'
          AND report_date <= (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + 2*LABEL_WINDOW_DAYS - 1} DAY)
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
        CASE WHEN impr_T1 > impr_baseline THEN 1 ELSE 0 END AS recovered_by_T1
    FROM future
"""
recovery_df = con.sql(future_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Data handling

#### Impute missing data

In [100]:
print("Before imputing:\n", X_raw.isna().sum())

Before imputing:
 client_hash_id                 0
content_hash_id                0
gsc_avg_position          115070
gsc_impressions                0
gsc_clicks                     0
keyword_char_count             0
url_char_count                 0
content_type                   0
search_volume              57071
competition                57071
main_intent                55417
category_count                 0
model_used                 84060
char_count                105346
days_since_last_update         0
days_since_created             0
dtype: int64


In [101]:
# Search volume and competition have the same missing count, combine into one missing flag
X_raw["has_keyword_data"] = X_raw["search_volume"].notna() & X_raw["competition"].notna()
X_raw["char_count_imputed"] = X_raw["char_count"].isna()
X_raw["ai_generated"] = X_raw["model_used"].notna()

X_raw["gsc_avg_position"] = X_raw["gsc_avg_position"].fillna(0) # 0 means no position data
X_raw["search_volume"] = X_raw["search_volume"].fillna(0)
X_raw["competition"] = X_raw["competition"].fillna(0)
X_raw["main_intent"] = X_raw["main_intent"].fillna("no_keyword")
X_raw["model_used"] = X_raw["model_used"].fillna("human")
X_raw["char_count"] = X_raw["char_count"].fillna(X_raw["char_count"].median())

In [102]:
print("After imputing:\n", X_raw.isna().sum())

After imputing:
 client_hash_id            0
content_hash_id           0
gsc_avg_position          0
gsc_impressions           0
gsc_clicks                0
keyword_char_count        0
url_char_count            0
content_type              0
search_volume             0
competition               0
main_intent               0
category_count            0
model_used                0
char_count                0
days_since_last_update    0
days_since_created        0
has_keyword_data          0
char_count_imputed        0
ai_generated              0
dtype: int64


### Model Building

In [126]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

declining_ids = gate_df.loc[gate_df["is_declining_at_T"] == 1, ["client_hash_id", "content_hash_id"]]

data = (X_raw.merge(declining_ids, on=["client_hash_id", "content_hash_id"], how="inner")
             .merge(recovery_df[["client_hash_id", "content_hash_id", "recovered_by_T1"]],
                     on=["client_hash_id", "content_hash_id"], how="inner"))

y = data.pop("recovered_by_T1")
groups = data["client_hash_id"]
categorical_text_cols = ["content_type", "main_intent", "model_used"]
X_full = pd.get_dummies(data.drop(columns=["client_hash_id", "content_hash_id"]), columns=categorical_text_cols)

def precision_at_k(scores, labels, k):
    k = min(k, len(scores))
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def run_group_kfold(X, y, groups, label):
    n_clients = groups.nunique()
    n_folds = min(5, n_clients)

    gkf = GroupKFold(n_splits=n_folds)
    fold_results = []
    coef_records = []
    importance_records = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
        X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

        logit = LogisticRegression(random_state=42, class_weight="balanced", max_iter=2000).fit(X_train_scaled, y_train)
        rf = RandomForestClassifier(max_depth=4, random_state=42, class_weight="balanced").fit(X_train, y_train)

        coef_records.append(pd.Series(logit.coef_[0], index=X_train.columns, name=f"fold_{fold}"))
        importance_records.append(pd.Series(rf.feature_importances_, index=X_train.columns, name=f"fold_{fold}"))

        stayed_broken = 1 - y_test.values
        logit_scores = 1 - logit.predict_proba(X_test_scaled)[:, 1]
        rf_scores = 1 - rf.predict_proba(X_test)[:, 1]

        fold_results.append({
            "fold": fold, "test_clients": groups.iloc[test_idx].nunique(), "test_rows": len(y_test),
            "recovered_rate": y_test.mean(),
            "logit_auc": roc_auc_score(y_test, logit.predict_proba(X_test_scaled)[:, 1]),
            "rf_auc": roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]),
            "logit_p@20": precision_at_k(logit_scores, stayed_broken, 20),
            "rf_p@20": precision_at_k(rf_scores, stayed_broken, 20),
            "logit_p@50": precision_at_k(logit_scores, stayed_broken, 50),
            "rf_p@50": precision_at_k(rf_scores, stayed_broken, 50),
            "base_rate": stayed_broken.mean(),
        })

    fold_df = pd.DataFrame(fold_results)
    fold_df["logit_p@20_lift"] = fold_df["logit_p@20"] - fold_df["base_rate"]
    fold_df["rf_p@20_lift"] = fold_df["rf_p@20"] - fold_df["base_rate"]
    fold_df["logit_p@50_lift"] = fold_df["logit_p@50"] - fold_df["base_rate"]
    fold_df["rf_p@50_lift"] = fold_df["rf_p@50"] - fold_df["base_rate"]

    metric_cols = ["logit_auc", "rf_auc", "logit_p@20", "logit_p@50", "rf_p@20", "rf_p@50", "logit_p@20_lift", "rf_p@20_lift", "logit_p@50_lift", "rf_p@50_lift"]
    summary = fold_df[metric_cols].agg(["mean", "std"]).T
    summary.columns = [f"{label}_mean", f"{label}_std"]

    coef_df = pd.concat(coef_records, axis=1)
    coef_summary = pd.DataFrame({"mean": coef_df.mean(axis=1), "std": coef_df.std(axis=1)}).sort_values("mean", key=abs, ascending=False)

    importance_df = pd.concat(importance_records, axis=1)
    importance_summary = pd.DataFrame({"mean": importance_df.mean(axis=1), "std": importance_df.std(axis=1)}).sort_values("mean", ascending=False)

    return {
        "n_clients": n_clients, "n_folds": n_folds, "n_features": X.shape[1],
        "fold_df": fold_df, "summary": summary,
        "coef_summary": coef_summary, "importance_summary": importance_summary,
    }

# --- run 1: all features ---
results_full = run_group_kfold(X_full, y, groups, label="full")

print(f"Full feature set: {results_full['n_features']} features")
print(results_full["summary"])
print(results_full["coef_summary"])

Full feature set: 27 features
                 full_mean  full_std
logit_auc         0.684086  0.126087
rf_auc            0.692027  0.107691
logit_p@20        0.790000  0.151658
logit_p@50        0.856000  0.121161
rf_p@20           0.910000  0.124499
rf_p@50           0.884000  0.145877
logit_p@20_lift   0.192420  0.176695
rf_p@20_lift      0.312420  0.175436
logit_p@50_lift   0.258420  0.165301
rf_p@50_lift      0.286420  0.191385
                                       mean       std
days_since_last_update            -0.534783  0.129807
model_used_gpt-5-mini             -0.450021  0.144093
model_used_gemini-3-flash-preview  0.364285  0.086534
char_count_imputed                 0.348852  0.102501
gsc_impressions                    0.185935  0.173417
char_count                        -0.175839  0.077625
has_keyword_data                   0.149887  0.072783
gsc_clicks                        -0.144320  0.127633
days_since_created                 0.129081  0.030244
gsc_avg_position       

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Raw precision looks strong on its own: logit p@20 0.790 ± 0.152, rf p@20 0.910 ± 0.124,
rf p@50 0.884 ± 0.146. Read without context, over 90% precision@20 looks like a very
strong model. However,

1. **Lift over each fold's own base rate tells a different story than the raw numbers.**
   Base rate varies a lot fold to fold, so raw precision@K is partly just reflecting how
   common "stayed broken" already was in that fold, not the model's own contribution.
   The best result (rf_p@20_lift) is, on average, about 31 percentage points better than
   base-rate guessing — a real improvement, but std (0.175) is still more than half the
   mean, so some folds show a much smaller gain than others.
2. **logit_p@20_lift has std (0.177) almost as large as its mean (0.192)** Close to half
   the folds could be showing close-to-no improvement over guessing, even though the
   average looks solid.

days_since_last_update (-0.535, std 0.130) is
the clear standout — small std relative to its mean.
model_used_gpt-5-mini (-0.450) and model_used_gemini-3-flash-preview (0.364) look like
the next-strongest effects, with char_count_imputed (0.349) close behind. content_type
and main_intent are both close to 0.

### Multicollinearity Check

In [84]:
!pip install phik --quiet

In [128]:
import phik

cols_to_check = [c for c in X_raw.columns if c not in ["client_hash_id", "content_hash_id"]]
phik_matrix = X_raw[cols_to_check].phik_matrix()

phik_flat = phik_matrix.copy()
np.fill_diagonal(phik_flat.values, np.nan)  # drop self-correlations (always 1.0)

phik_pairs = phik_flat.unstack().dropna().sort_values(ascending=False)
phik_pairs = phik_pairs[phik_pairs.index.get_level_values(0) < phik_pairs.index.get_level_values(1)]  # drop mirror duplicates

threshold = 0.5
strong_pairs = phik_pairs[phik_pairs > threshold]

# --- fix: flatten the MultiIndex into plain columns before printing ---
strong_pairs = strong_pairs.reset_index()
strong_pairs.columns = ["col1", "col2", "phik"]
strong_pairs = strong_pairs.sort_values("phik", ascending=False)

print(strong_pairs.to_string(index=False))

interval columns not set, guessing: ['gsc_avg_position', 'gsc_impressions', 'gsc_clicks', 'keyword_char_count', 'url_char_count', 'search_volume', 'competition', 'category_count', 'char_count', 'days_since_last_update', 'days_since_created']
                  col1                   col2     phik
          ai_generated             model_used 1.000000
      has_keyword_data     keyword_char_count 0.999283
    char_count_imputed             model_used 0.983298
          ai_generated     char_count_imputed 0.967512
      has_keyword_data            main_intent 0.866435
    keyword_char_count            main_intent 0.837076
    char_count_imputed     days_since_created 0.774066
          content_type     keyword_char_count 0.752969
      has_keyword_data             model_used 0.689322
            gsc_clicks        gsc_impressions 0.683798
    days_since_created       has_keyword_data 0.650822
          content_type            main_intent 0.648900
    days_since_created            main_inte

#### Key Findings
1. `ai_generated` / `model_used` (1.00) and `has_keyword_data` / `keyword_char_count`
  (0.999) are expected: both flags are directly derived from their source
  column's missingness. Not redundant.

2. `char_count_imputed` / `model_used` (0.98) and `ai_generated` / `char_count_imputed`
  (0.97): char_count's missingness is concentrated in specific model_used
  categories. Not random.

3. `main_intent` / `keyword_char_count` (0.84) and
  `content_type` / `keyword_char_count` (0.75): both driven by intent
  and format largely determining keyword length.

4. `gsc_impressions` / `gsc_clicks` (0.68): Create a
  derived `ctr = gsc_clicks / gsc_impressions` to represent how well a page
  converts.

5. `days_since_created` associate broadly (0.40-0.77)
  with nearly every other column from (2), potentially tracking which content-production era a page came from.

6. `days_since_last_update` shows moderate correlation
  (0.36-0.51) with model_used, has_keyword_data, main_intent, url_char_count,
  char_count_imputed.

After checking correlations, a cluster of features: model_used,
char_count, char_count_imputed, and ai_generated may indicate one signal counted four times. That raises the question: is content_type actually near-zero, or is this cluster hiding it?

In [129]:
# Revision 1:  drop char_count/char_count_imputed, keyword_char_count, gsc_clicks (replaced with ctr)
X_r1["ctr"] = data["gsc_clicks"] / data["gsc_impressions"].replace(0, np.nan)
X_r1["ctr"] = X_r1["ctr"].fillna(0)
drop_r1 = ["char_count", "char_count_imputed", "keyword_char_count", "gsc_clicks"]

X_r1 = pd.get_dummies(
    data.drop(columns=["client_hash_id", "content_hash_id"] +
              [c for c in drop_r1 if c in data.columns]),
    columns=categorical_text_cols  # keep model_used in the encoding list -- don't exclude it
)

results_r1 = run_group_kfold(X_r1, y, groups, label="r1")
print(f"Revision 1: {X_r1.shape[1]} features (dropped {len(drop_r1)}: {drop_r1})")

comparison = pd.concat([
    results_full["summary"],
    results_r1["summary"],
], axis=1)

print(comparison.round(3))

print(results_r1["coef_summary"].head(10))

Revision 1: 23 features (dropped 4: ['char_count', 'char_count_imputed', 'keyword_char_count', 'gsc_clicks'])
                 full_mean  full_std  r1_mean  r1_std
logit_auc            0.684     0.126    0.675   0.130
rf_auc               0.692     0.108    0.701   0.123
logit_p@20           0.790     0.152    0.840   0.139
logit_p@50           0.856     0.121    0.868   0.120
rf_p@20              0.910     0.124    0.880   0.144
rf_p@50              0.884     0.146    0.872   0.157
logit_p@20_lift      0.192     0.177    0.242   0.187
rf_p@20_lift         0.312     0.175    0.282   0.184
logit_p@50_lift      0.258     0.165    0.270   0.170
rf_p@50_lift         0.286     0.191    0.274   0.207
                                       mean       std
days_since_last_update            -0.629127  0.101351
model_used_gpt-5-mini             -0.443530  0.158415
days_since_created                 0.275285  0.041945
model_used_human                   0.197494  0.052014
ai_generated              

In [130]:
# Revision 2: Now also drop model_used/ai_generated,
# since they clearly did NOT just ride on char_count's coattails
drop_r2 = ["char_count", "char_count_imputed", "keyword_char_count", "gsc_clicks", "model_used", "ai_generated"]
categorical_r2 = [c for c in categorical_text_cols if c != "model_used"]

X_r2 = pd.get_dummies(
    data.drop(columns=["client_hash_id", "content_hash_id"] +
              [c for c in drop_r2 if c in data.columns]),
    columns=categorical_r2
)

X_r2["ctr"] = data["gsc_clicks"] / data["gsc_impressions"].replace(0, np.nan)
X_r2["ctr"] = X_r2["ctr"].fillna(0)

results_r2 = run_group_kfold(X_r2, y, groups, label="r2")
print(f"Revision 2: {X_r2.shape[1]} features (dropped {len(drop_r2)}: {drop_r2})")

comparison = pd.concat([
    comparison,
    results_r2["summary"],
], axis=1)

print(comparison.round(3))

print(results_r2["coef_summary"].head(10))

Revision 2: 17 features (dropped 6: ['char_count', 'char_count_imputed', 'keyword_char_count', 'gsc_clicks', 'model_used', 'ai_generated'])
                 full_mean  full_std  r1_mean  r1_std  r2_mean  r2_std
logit_auc            0.684     0.126    0.675   0.130    0.670   0.083
rf_auc               0.692     0.108    0.701   0.123    0.676   0.078
logit_p@20           0.790     0.152    0.840   0.139    0.870   0.135
logit_p@50           0.856     0.121    0.868   0.120    0.896   0.098
rf_p@20              0.910     0.124    0.880   0.144    0.930   0.097
rf_p@50              0.884     0.146    0.872   0.157    0.928   0.083
logit_p@20_lift      0.192     0.177    0.242   0.187    0.272   0.220
rf_p@20_lift         0.312     0.175    0.282   0.184    0.332   0.139
logit_p@50_lift      0.258     0.165    0.270   0.170    0.298   0.185
rf_p@50_lift         0.286     0.191    0.274   0.207    0.330   0.109
                                  mean       std
days_since_last_update       -

In [133]:
# Check with RFECV

from sklearn.feature_selection import RFECV

# RFECV needs a single train/test structure per fold internally --
# use GroupKFold as the cv strategy so it respects client-grouping
rfecv = RFECV(
    estimator=LogisticRegression(class_weight="balanced", max_iter=2000),
    step=1,
    cv=GroupKFold(n_splits=5),
    scoring="roc_auc",
    min_features_to_select=3,
)

# needs scaled data, and groups passed explicitly
scaler = StandardScaler()
X_r2_scaled = pd.DataFrame(scaler.fit_transform(X_r2), columns=X_r2.columns)
rfecv.fit(X_r2_scaled, y, groups=groups)

print(f"Optimal number of features: {rfecv.n_features_}")
print(f"Selected: {X_r2.columns[rfecv.support_].tolist()}")

Optimal number of features: 17
Selected: ['gsc_avg_position', 'gsc_impressions', 'url_char_count', 'search_volume', 'competition', 'category_count', 'days_since_last_update', 'days_since_created', 'has_keyword_data', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_commercial', 'main_intent_informational', 'main_intent_navigational', 'main_intent_no_keyword', 'main_intent_transactional', 'ctr']


This confirms the suspicion: content_type's near-zero coefficient
(0.010, std 0.121) in the full model was the confound cluster masking it, not a real
finding. Once model_used/char_count are removed (Revision 2), content_type becomes a real effect (0.195, std 0.118). Feedly articles recover less than keyword articles,
holding staleness and position constant. days_since_last_update stays the most trusted
signal throughout, strengthening rather than weakening as the confound is removed
(-0.535 full → -0.881 R2), which is the opposite of what you'd expect if it were
itself an artifact.

RFECV's selected feature count also selects 17 features with 5 folds, confirming results from Revisions 1 and 2.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.